In [1]:
import json
import pandas as pd
import numpy as np
import math

# To extract heading from quaternion (orientation)
def quaternion_to_euler(w, x, y, z):
    """Convert quaternion (w, x, y, z) to Euler angles (roll, pitch, yaw)."""
    t0 = +2.0 * (w * x + y * z)
    t1 = +1.0 - 2.0 * (x * x + y * y)
    roll_x = math.atan2(t0, t1)
    
    t2 = +2.0 * (w * y - z * x)
    t2 = +1.0 if t2 > +1.0 else t2
    t2 = -1.0 if t2 < -1.0 else t2
    pitch_y = math.asin(t2)
    
    t3 = +2.0 * (w * z + x * y)
    t4 = +1.0 - 2.0 * (y * y + z * z)
    yaw_z = math.atan2(t3, t4)
    
    return roll_x, pitch_y, yaw_z


def parse_json_file(file_path):
    data_by_topic = {}  # Dictionary to store data organised by topic

    with open(file_path, 'r') as file:
        for line in file:
            try:
                # Parse each JSON object
                obj = json.loads(line.strip())

                # Organise data by topic
                obj_topic = obj.get("topic_name", "Unknown")
                if obj_topic not in data_by_topic:
                    data_by_topic[obj_topic] = []
                data_by_topic[obj_topic].append(obj)

            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}")

    return data_by_topic



In [2]:
file_path = r'Test 1/first_test_log.json'  # Replace with your json log file
data_by_topic = parse_json_file(file_path)

aruco_path = r'Test 1/first_test_aruco.csv'  # Replace with your ArUco csv file

print("\nData organized by topic:")
for topic, items in data_by_topic.items():
    print(f"{topic}: {len(items)} entries")


Data organized by topic:
/true_wheel_speeds: 417 entries
/aruco: 36 entries
/lidar: 36 entries
/est_pose: 361 entries
/wheel_speeds_cmd: 361 entries


In [3]:
# est_pose
est_pose_timestamps = [item['message']['header']['stamp'] for item in data_by_topic['/est_pose']]
est_pose_northings = [item['message']['pose']['position']['x'] for item in data_by_topic['/est_pose']]
est_pose_eastings = [item['message']['pose']['position']['y'] for item in data_by_topic['/est_pose']]
est_pose_headings = [(quaternion_to_euler(item['message']['pose']['orientation']['w'], 
                                         item['message']['pose']['orientation']['x'], 
                                         item['message']['pose']['orientation']['y'], 
                                         item['message']['pose']['orientation']['z'])[2])% (2*np.pi)
                     for item in data_by_topic['/est_pose']]


# Wheel speeds
true_wheel_speeds_timestamps = [item['message']['header']['stamp'] for item in data_by_topic['/true_wheel_speeds']]
true_wheel_speeds = [item['message']['vector']['x'] for item in data_by_topic['/true_wheel_speeds']]
wheel_speeds_cmd_timestamps = [item['message']['header']['stamp'] for item in data_by_topic['/wheel_speeds_cmd']]
wheel_speeds_cmd = [item['message']['vector']['x'] for item in data_by_topic['/wheel_speeds_cmd']]


# aruco csv
df = pd.read_csv(aruco_path)   
start_time = np.min(true_wheel_speeds_timestamps)
end_time = np.max(true_wheel_speeds_timestamps)
filtered_data = df.loc[(df['epoch [s]'] >= start_time) & (df['epoch [s]'] <= end_time)]
filtered_data = filtered_data[filtered_data.iloc[:, -1] == True]

aruco_timestamps = filtered_data['epoch [s]']
aruco_northings = filtered_data['x [m]']
aruco_eastings = filtered_data['y [m]']
aruco_headings = np.deg2rad(filtered_data['yaw [deg]'] % (360))


### Plot pose

In [4]:
import matplotlib.pyplot as plt
%matplotlib qt

plt.figure(figsize=(10, 8))
plt.scatter(aruco_eastings, aruco_northings, label="Aruco", marker="x", color="blue")
plt.plot(est_pose_eastings, est_pose_northings, label="Estimated Pose", marker=".", color="green")
# plt.plot(lidar_eastings, lidar_northings, label="Lidar", marker="s")

plt.xlabel("Eastings (m)")
plt.ylabel("Northings(m)")
plt.legend()
plt.title("Env. Frame Locations")
plt.grid()
plt.show()

### Plot wheels speeds

In [5]:
# Ensure timestamps are float
true_wheel_speeds_timestamps = [float(ts) for ts in true_wheel_speeds_timestamps]
wheel_speeds_cmd_timestamps = [float(ts) for ts in wheel_speeds_cmd_timestamps]

plt.figure(figsize=(10, 6))

plt.plot(true_wheel_speeds_timestamps, true_wheel_speeds, label="Measured Wheel Speeds", marker=".", color="orange")
plt.plot(wheel_speeds_cmd_timestamps, wheel_speeds_cmd, label="Commanded Wheel Speeds", marker=".", color="green")

plt.xlabel("Time (s)")
plt.ylabel("Wheel Speed (rad/s)")
plt.legend()
plt.title("Measured and Commanded Wheel Speeds")
plt.grid()
plt.show()

### Plot heading

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(aruco_timestamps, aruco_headings, label="Aruco Heading", marker="x", color="blue")
plt.plot(est_pose_timestamps, est_pose_headings, label="Estimated Pose Heading", marker=".", color="green")


plt.xlabel("Time (s)")
plt.ylabel("Heading (degrees)")
plt.legend()
plt.title("Headings: Aruco vs Estimated Pose")
plt.grid()
plt.show()

: 